
# 練習問題: 実行をGPUに移す

## 目標

`#pragma omp target` (Fortran では `!$omp target` ... `!$omp end target`) を1つ挿入するだけで, 1つの文の実行をデバイス(GPU)に移せることを体験する.

## 課題

`target_offload.cpp` (または `target_offload.f90`) は, 現状ではホスト(CPU)上で1行のメッセージを表示する.
コメント `TODO` の指示に従って **OpenMP の指示行を1つ追加** し, その `printf` (`print`) 文がGPU上で実行されるようにせよ.

- C++: メッセージを表示するブロック `{ ... }` の直前に `#pragma omp target` を1行加える.
- Fortran: `print` 文を `!$omp target` と `!$omp end target` で囲む.

それ以外のコードを変更する必要はない.

## コンパイルと実行

```
# C++
nvc++ -mp=gpu target_offload.cpp -o target_offload.exe

# Fortran
nvfortran -mp=gpu target_offload.f90 -o target_offload.exe
```

GPUは計算ノードにのみ搭載されているので, `%%bash_submit` でジョブとして投入して実行する.
必要に応じて rscgrp や elapse を指定せよ.

```
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:10:00

OMP_TARGET_OFFLOAD=MANDATORY ./target_offload.exe
```

## 期待される結果

メッセージが表示される.

```
hello from the device
```

`printf` / `print` の出力結果自体はCPUで実行してもGPUで実行しても同じなので, 見た目では区別がつかない. そこで `OMP_TARGET_OFFLOAD` を切り替えて挙動を比べてみよ.

- `OMP_TARGET_OFFLOAD=MANDATORY` ... GPUでの実行を強制する. `target` の挿入を忘れていてもエラーにはならない(`target` 領域がなければGPUを使わないため)が, GPUがないログインノードで `target` 付きで実行するとエラーになる.
- `OMP_TARGET_OFFLOAD=DISABLED` ... 必ずCPUで実行する.

`target` を正しく挿入したうえで, 計算ノードで `MANDATORY` を指定して(GPUで)実行できることを確認せよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ target_offload.cpp
#include <cstdio>
#include <omp.h>

int main() {
  // TODO: 下のブロックの直前に #pragma omp target を1行追加し, printf をデバイス(GPU)上で実行させよ.
  {
    printf("hello from the device\n");
  }
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=gpu target_offload.cpp -o target_offload_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./target_offload_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ target_offload.f90
program target_offload
  use omp_lib
  ! TODO: 下の print を !$omp target ... !$omp end target で囲み, デバイス(GPU)上で実行させよ.
  print "(a)", "hello from the device"
  ! TODO: 上で始めた target 領域を閉じる (!$omp end target).
end program target_offload

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=gpu target_offload.f90 -o target_offload_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./target_offload_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:target_offload.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:target_offload.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:target_offload.cpp}